# HarmonicSlot-TF v3 (`bf16`): Unified Single-Model 48 kHz Separator
1. Upload `model.py`, `loss.py`, `train.py`, `prepare_data.py` (from `E:\AudioSep_Dataset_48k\gdrive_upload\`) into `/content/drive/MyDrive/audiosep_data_48k/`.
2. **Cell 1** mounts Google Drive and checks your A100 GPU.
3. **Cell 2** copies the scripts into `/content/AudioSep`, prints SHA256 hashes, and extracts cached `.tar` audio datasets directly into `/content/data48` (automatically preferring `music_hq48.tar` / `amb_48k.tar` if present).
4. **Cell 2b (Optional, one-time):** Streams up to `60` audited vocal-free 48 kHz stereo MUSDB18-HQ tracks (~1.3 GB) + verified speech/music-free 48 kHz DEMAND ambience (~0.8 GB) and caches `music_hq48.tar` & `amb_48k.tar` on Drive.
5. **Cell 3 (Phase 1, `0 -> 20,000`)** trains `HarmonicSlotTF v3` (`dim=384, enc=8, dec=4`, RoPE + Decoder Inter-Source + EMA `0.999`, `bf16`, `--save-bf16`). Inspect `[VAL step 5000]` before continuing.
6. **Cell 4 (Phase 2, `20,000 -> 28,000`)** runs paired `6s -> 6s` (`1.5s` overlap) stateful cross-chunk `prev_slots` continuity training with EMA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install soundfile safetensors transformers
!nvidia-smi

In [ ]:
import os, shutil, subprocess
from pathlib import Path

os.makedirs('/content/AudioSep', exist_ok=True)
os.makedirs('/content/data48', exist_ok=True)

code_candidates = [
    Path('/content/drive/MyDrive/audiosep_data_48k'),
    Path('/content/drive/MyDrive/AudioSep'),
    Path('/content/drive/MyDrive/Colab Notebooks/AudioSep'),
    Path('/content/drive/MyDrive/Colab Notebooks/audiosep_data'),
]
for cand in code_candidates:
    if cand.exists() and (cand / 'train.py').exists():
        print('Copying scripts from:', cand)
        for py in sorted(cand.glob('*.py')):
            shutil.copy2(py, f'/content/AudioSep/{py.name}')
        break

!sha256sum /content/AudioSep/loss.py /content/AudioSep/model.py /content/AudioSep/train.py /content/AudioSep/prepare_data.py
%cd /content/AudioSep

tar_dir = Path('/content/drive/MyDrive/audiosep_data_48k')
if tar_dir.exists() and (tar_dir / 'speech.tar').exists():
    tar_sources = [
        ('speech', ['speech.tar']),
        ('rir', ['rir.tar']),
        ('music', ['music_hq48.tar', 'music.tar']),
        ('amb', ['amb_48k.tar', 'amb.tar']),
    ]
    for sub, candidates in tar_sources:
        picked = next((tar_dir / c for c in candidates if (tar_dir / c).exists()), None)
        if picked is None:
            continue
        dst_sub = Path('/content/data48') / sub
        if dst_sub.exists() and any(dst_sub.iterdir()):
            if picked.name in ('music_hq48.tar', 'amb_48k.tar') and not list(dst_sub.glob('*.flac')):
                shutil.rmtree(dst_sub)
            else:
                continue
        print(f'Extracting cached {picked.name} -> /content/data48/{sub}...')
        subprocess.run(['tar', '-xf', str(picked), '-C', '/content/data48'], check=True)
    for sub in ('speech', 'music', 'amb', 'rir'):
        p = Path('/content/data48') / sub
        cnt = sum(1 for f in p.rglob('*') if f.is_file()) if p.exists() else 0
        print(f'/content/data48/{sub}: {cnt} files')
else:
    !python prepare_data.py --preset full_48k --out /content/data48 --verify --pack-tar /content/drive/MyDrive/audiosep_data_48k

In [ ]:
# OPTIONAL (one-time upgrade, ~5-10 min): Streams up to 60 MUSDB18-HQ tracks (~1.3 GB) + 8 speech-free 48 kHz DEMAND rooms (~0.8 GB), then runs strict Silero VAD + AudioSet AST purity audit.
# If you skip this cell, Cell 3 & Cell 4 still run normally using your existing music.tar & amb.tar.
from pathlib import Path
tar_dir = Path('/content/drive/MyDrive/audiosep_data_48k')
if (tar_dir / 'music_hq48.tar').exists() and (tar_dir / 'amb_48k.tar').exists():
    print('Full-band music_hq48.tar and amb_48k.tar already cached on Drive — skipping download.')
else:
    !python -u /content/AudioSep/prepare_data.py --preset upgrade_bg_48k --max-music 60 --audit --out /content/data48 --verify --pack-tar /content/drive/MyDrive/audiosep_data_48k

In [ ]:
# Phase 1: HarmonicSlotTF v3 (bf16, 0 -> 20,000 steps; check [VAL step 5000] stop-loss gate)
from pathlib import Path
p1_dir = Path('/content/drive/MyDrive/audiosep_48k_v3_86m')
rflag = f'--resume {p1_dir / "last.pt"}' if (p1_dir / 'last.pt').exists() else ''
print('Phase-1 flags:', rflag or '(fresh start from step 0)')
!python -u /content/AudioSep/train.py --arch v3 --mode purity_queries --precision bf16 --save-bf16 \
    --sr 48000 --channels 2 --n-fft 2048 --hop-length 256 \
    --dim 384 --n-enc-layers 8 --n-dec-layers 4 \
    --use-inpaint --use-enhancer --grad-checkpoint \
    --batch-size 4 --grad-accum 1 --seg-seconds 4.0 --val-seg-seconds 4.0 \
    --p-3spk 0.5 --hard-lang-prob 0.4 --full-overlap-prob 0.5 --mono-mix-prob 0.5 --fast-pit \
    --bleed-weight 1.5 --tf-bleed-weight 0.5 --cross-bleed-weight 1.0 \
    --activity-weight 0.3 --mix-cons-weight 0.1 --ortho-weight 0.2 \
    --lr 3e-4 --ema-decay 0.999 --steps 20000 --eval-every 500 --num-workers 4 \
    --speech-dir /content/data48/speech --music-dir /content/data48/music \
    --amb-dir /content/data48/amb --rir-dir /content/data48/rir \
    --save-dir /content/drive/MyDrive/audiosep_48k_v3_86m {rflag}

In [ ]:
# Phase 2: Paired 4s/6s Stateful Cross-Chunk + Gap Continuity Training (17,000 -> 21,000 steps)
from pathlib import Path
p1_dir = Path('/content/drive/MyDrive/audiosep_48k_v3_86m')
p2_dir = Path('/content/drive/MyDrive/audiosep_48k_v3_86m_p2')
p2_dir.mkdir(parents=True, exist_ok=True)
if (p2_dir / 'last.pt').exists():
    rflag = f'--resume {p2_dir / "last.pt"}'
else:
    src = (p1_dir / 'best.pt') if (p1_dir / 'best.pt').exists() else (p1_dir / 'last.pt')
    assert src.exists(), f'No Phase-1 checkpoint found in {p1_dir}'
    rflag = f'--resume {src} --finetune'
print('Phase-2 flags:', rflag)
!python -u /content/AudioSep_p2/train.py --arch v3 --mode purity_queries --precision bf16 --save-bf16 \
    --sr 48000 --channels 2 --n-fft 2048 --hop-length 256 \
    --dim 384 --n-enc-layers 8 --n-dec-layers 4 \
    --use-inpaint --use-enhancer --grad-checkpoint \
    --batch-size 2 --grad-accum 1 --seg-seconds 4.0 --val-seg-seconds 4.0 \
    --streaming-prob 0.85 --stream-chunk-seconds 6.0 --stream-overlap-seconds 1.5 \
    --p-3spk 0.5 --hard-lang-prob 0.4 --full-overlap-prob 0.5 --mono-mix-prob 0.5 --fast-pit \
    --bleed-weight 1.5 --tf-bleed-weight 0.5 --cross-bleed-weight 1.0 \
    --activity-weight 0.3 --mix-cons-weight 0.1 --ortho-weight 0.2 \
    --lr 7e-5 --ema-decay 0.999 --steps 21000 --eval-every 250 --num-workers 4 \
    --speech-dir /content/data48/speech --music-dir /content/data48/music \
    --amb-dir /content/data48/amb --rir-dir /content/data48/rir \
    --save-dir /content/drive/MyDrive/audiosep_48k_v3_86m_p2 {rflag} 2>&1 | tee -a /content/drive/MyDrive/audiosep_48k_v3_86m_p2/train_p2.log